# Ekspor Gelombang-7 -- TRAIN + VAL/TEST sekaligus utk 5 Kelas Lemah (target volume piksel besar)

## Konteks
Setelah gelombang-5/6 (train-only), rasio pool minoritas sudah 83:8.5:8.3 (dekat 80:10:10), tapi
mIoU kelas lemah masih di kisaran 0.4-0.5 (lihat run fine-tune model_1 di Kaggle). **Keputusan
USER**: kejar volume PIKSEL (bukan jumlah patch) jauh lebih besar -- target eksplisit:

- **TRAIN-ONLY** (gelombang-7 train): minimal **~20.000.000 piksel per kelas lemah** (tambahan,
  di atas yang sudah ada dari gel.5+gel.6).
- **VAL+TEST** (gelombang-7 valtest, gabungan): minimal **~10.000.000 piksel per kelas lemah**
  (tambahan, di atas yang sudah ada dari gel.4).

**Catatan jujur soal Tambang**: 20 juta piksel @ resolusi 10m = ~2.000 km² luasan TAMBANG NYATA
(bukan luas bbox) -- 9 mega-tambang raksasa di gelombang-6 (bbox 30-60km tiap satu) cuma
hasilkan ~3 juta piksel Tambang AKTUAL gabungan. Target ini kemungkinan besar TIDAK tercapai
murni dari data tambang dunia nyata utk kelas Tambang spesifik -- region sebanyak & sebesar
mungkin tetap dikumpulkan, tapi GATE (bagian T2) akan menunjukkan estimasi realistis sebelum
ekspor. 4 kelas lain (Sawit, Pertanian Lain, Lahan Terbuka, Permukiman) jauh lebih realistis
mencapai target krn luasannya kontinu di alam (bukan polygon kecil tersebar spt tambang).

## Desain: TRAIN dan VAL/TEST dari region yang BERBEDA TOTAL (bukan split dari region yang sama)
- `TRAIN_GEL7`: 16 region BARU (8 Tambang + 2 tiap 4 kelas lain), KHUSUS train -- tidak pernah
  disentuh val/test, jadi tidak ada risiko leakage train->val/test dari sisi region ini.
- `VALTEST_GEL7`: 12 region BARU LAIN (4 Tambang + 2 tiap 4 kelas lain), KHUSUS val/test --
  setiap kelas punya >=2 region supaya bisa displit PER-REGION (bukan per-patch) antara val
  dan test -- setengah region pertama -> val, setengah region terakhir -> test. Ini menghindari
  leakage spasial val<->test (region yang sama tidak pernah muncul di keduanya).

Semua 28 region baru sudah diverifikasi **0% overlap geografis** dgn seluruh 117 bbox yang
sudah pernah dipakai gelombang 1-6, dan 0% overlap antar-sesama region baru (cross-check T1).

## Sumber region baru (negara/wilayah yang belum pernah dipakai gelombang manapun)
- **Tambang train**: Uzbekistan (Muruntau, emas terbuka terbesar di dunia), Rusia (Norilsk),
  India (Jharia coalfield), Indonesia (Bangka, timah aluvial), Swedia (Kiirunavaara), Tiongkok
  (Dexing), Kazakhstan (Zhezkazgan), Mauritania (Zouerate, bijih besi).
- **Tambang valtest**: Ghana (Obuasi), Mali (Loulo-Gounkoto), Peru (Cerro de Pasco), Bolivia
  (Cerro Rico).
- **4 kelas lain**: blok besar baru di Kaltara/Bengkulu/Sultra/Sulbar (sawit), Kalbar/Sumut/Aceh/
  Sumsel (lahan terbuka), Sumbar/NTT/Sulut/NTB (pertanian lain), Batam/Tarakan/Tegal-Pekalongan/
  Cilegon (permukiman).

## Yang TIDAK disentuh
`Bahan_Training_Fix` (Papua lama, FROZEN), `ForestWatch_Patches_TransferEval` (gel.4, val_new/
test_new FROZEN, train_new tetap dipakai), `ForestWatch_Patches_TrainOnlyGel5/Gel6` (tetap
dipakai, tidak diubah), `train_model_1/2/3.ipynb`, `compare_and_select_best_model.ipynb`.

## Output akhir
Cell T8 menggabungkan SEMUA (Papua lama + gel.4 train/val/test + gel.5 train + gel.6 train +
gel.7 train + gel.7 val_new7/test_new7), distribusi piksel final per split, paket ulang `.tar`
ke `Bahan_Training_Fix_Combined_v4/` (gantikan v3).


In [ ]:
# === COLAB SETUP (clone pertama kali / PULL sesi berikutnya) ===
%cd /content
!git -C fw_repo pull -q || git clone --depth 1 https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo
%cd fw_repo
!pip install -q -e ".[gee,gis,ml]"

import sys, importlib
if '/content/fw_repo/model/src' not in sys.path:
    sys.path.insert(0, '/content/fw_repo/model/src')
for mod in list(sys.modules.keys()):
    if mod == 'forestwatch' or mod.startswith('forestwatch.'):
        del sys.modules[mod]
importlib.invalidate_caches()

import forestwatch
print(f'forestwatch v{forestwatch.__version__}')


In [ ]:
# === MOUNT GOOGLE DRIVE ===
from google.colab import drive
drive.mount('/content/drive')
print('Drive ter-mount.')


In [ ]:
# === KONFIGURASI PATH (folder BARU, terpisah dari gelombang sebelumnya) ===
from pathlib import Path
from forestwatch.utils.io import save_json, load_json
from forestwatch.config import load_config

DRIVE_ROOT = Path('/content/drive/MyDrive/Satria Data 3.0')
TILES_GEL7_TRAIN     = DRIVE_ROOT / 'ForestWatch_Tiles_Gel7Train'
PATCHES_GEL7_TRAIN   = DRIVE_ROOT / 'ForestWatch_Patches_Gel7Train'
TILES_GEL7_VALTEST   = DRIVE_ROOT / 'ForestWatch_Tiles_Gel7ValTest'
PATCHES_GEL7_VALTEST = DRIVE_ROOT / 'ForestWatch_Patches_Gel7ValTest'
DIST_DIR = DRIVE_ROOT / 'Distribution_Reports'

for d in (TILES_GEL7_TRAIN, PATCHES_GEL7_TRAIN, TILES_GEL7_VALTEST, PATCHES_GEL7_VALTEST, DIST_DIR):
    d.mkdir(parents=True, exist_ok=True)

cfg = load_config()
print('Folder BARU (gelombang-7):')
print(' -', TILES_GEL7_TRAIN)
print(' -', PATCHES_GEL7_TRAIN)
print(' -', TILES_GEL7_VALTEST)
print(' -', PATCHES_GEL7_VALTEST)


In [ ]:
# === AUTH GEE + Periode T2 ===
import ee
from forestwatch.gee.auth import init_ee

init_ee(project=cfg['project']['gee_project_id'])
T2 = cfg['periods']['t2']
print(f'GEE siap. Periode label T2={T2}.')


In [ ]:
# === Bagian T1 -- TRAIN_GEL7 + VALTEST_GEL7 (region BARU TOTAL, beda dari gelombang 1-6) ===
# ALREADY_USED_BBOXES = union seluruh bbox gelombang 1-6 (117 entri).
ALREADY_USED_BBOXES = {
    # --- gel 1-3 ---
    'kaltim_sangatta':        (117.31,   0.31, 117.79,   0.79),
    'kaltim_sangatta_gel3':   (117.40,   0.25, 117.90,   0.75),
    'kalsel_tanahbumbu':      (115.41,  -3.69, 115.89,  -3.21),
    'sulteng_morowali':       (121.946, -2.974, 122.234, -2.686),
    'sumbawa_batuhijau':      (116.729, -9.062, 117.001, -8.838),
    'chile_chuquicamata':     (-68.996, -22.416, -68.804, -22.224),
    'chile_escondida':        (-69.156, -24.369, -68.964, -24.161),
    'usa_bingham':            (-112.236, 40.440, -112.044, 40.600),
    'aus_kalgoorlie':         (121.414, -30.850, 121.606, -30.690),
    'aus_huntervalley':       (150.725, -32.686, 151.125, -32.334),
    'australia_huntervalley_gel3': (150.70, -32.75, 151.20, -32.25),
    'ger_hambach':            (6.412,   50.827,   6.668,  51.003),
    'halmahera_wedabay':      (127.62,  -0.75, 128.32,  -0.05),
    'aus_pilbara':            (119.35, -23.75, 120.15, -22.95),
    'usa_powderriver':        (-105.95,  43.85, -105.05,  44.75),
    'kalsel_adaro':           (115.10,  -2.50, 115.80,  -1.80),
    'brazil_carajas':         (-50.60,  -6.40, -49.90,  -5.70),
    'brazil_carajas_gel3':    (-50.40,  -6.20, -50.00,  -5.80),
    'papua_grasberg':         (137.00,  -4.30, 137.25,  -4.00),
    'peru_antamina':          (-77.15,  -9.62, -76.95,  -9.42),
    'safrica_witbank':        (29.10,  -26.00,  29.40,  -25.75),
    'sumsel_tanjungenim':     (103.65,  -3.90, 104.10,  -3.45),
    'safrica_sishen':         (22.70,  -28.10,  23.20,  -27.60),
    'canada_athabasca':       (-111.75,  56.95, -111.30,  57.40),
    'australia_bowenbasin':   (148.00, -22.00, 148.50, -21.50),
    'jabodetabek': (106.70, -6.35, 106.95, -6.10), 'bandung': (107.55, -6.97, 107.70, -6.85),
    'surabaya': (112.68, -7.32, 112.82, -7.20), 'medan': (98.62, 3.52, 98.74, 3.64),
    'makassar': (119.40, -5.18, 119.52, -5.08), 'palembang': (104.62, -3.08, 104.88, -2.85),
    'semarang': (110.32, -7.05, 110.52, -6.90), 'denpasar': (115.13, -8.75, 115.32, -8.55),
    'balikpapan': (116.78, -1.32, 116.98, -1.12), 'pekanbaru': (101.35, 0.42, 101.58, 0.62),
    'papua_jayapura': (140.66, -2.62, 140.78, -2.50), 'papua_merauke': (140.36, -8.52, 140.48, -8.40),
    'papua_timika': (136.84, -4.58, 136.96, -4.46), 'papua_sorong': (131.22, -0.92, 131.34, -0.80),
    'papua_biak': (136.04, -1.20, 136.16, -1.08), 'maluku_ambon': (128.14, -3.72, 128.26, -3.60),
    'ntt_kupang': (123.54, -10.22, 123.70, -10.08), 'bogor_depok': (106.65, -6.75, 107.05, -6.35),
    'padang': (100.20, -1.15, 100.60, -0.75), 'banjarmasin': (114.40, -3.50, 114.80, -3.10),
    'pontianak': (109.10, -0.20, 109.50, 0.20), 'manado': (124.65, 1.30, 125.05, 1.70),
    'yogyakarta': (110.25, -8.00, 110.60, -7.65),
    'riau_pelalawan': (101.40, 0.20, 101.70, 0.50), 'sumut': (99.50, 2.00, 99.80, 2.30),
    'kalbar': (109.50, 0.00, 109.80, 0.30), 'kalteng': (112.50, -2.20, 112.80, -1.90),
    'sumsel_musibanyuasin': (103.80, -2.85, 104.20, -2.45),
    'riau_rokanhilir': (100.90, 1.50, 101.30, 1.90), 'riau_kampar': (101.00, 0.00, 101.40, 0.40),
    'riau_indragirihilir': (102.80, -0.70, 103.20, -0.30),
    'sumut_labuhanbatu': (99.90, 1.85, 100.30, 2.25), 'kalbar_ketapang': (110.20, -1.80, 110.60, -1.40),
    'kaltim_tepitambang': (117.30, 0.30, 117.55, 0.55), 'kalteng_pascabakar': (113.50, -2.50, 113.80, -2.20),
    'pantura_indramayu': (108.20, -6.45, 108.45, -6.25), 'lampung_ladang': (105.20, -5.10, 105.45, -4.90),
    # --- gel 4 ---
    'chile_lospelambres': (-70.65, -31.85, -70.35, -31.60), 'peru_cerroverde': (-71.65, -16.62, -71.42, -16.42),
    'usa_morenci': (-109.48, 32.98, -109.25, 33.18), 'australia_mountisa': (139.40, -20.80, 139.62, -20.62),
    'chile_losbronces': (-70.40, -33.25, -70.20, -33.05), 'usa_climax': (-106.30, 39.28, -106.05, 39.48),
    'peru_toquepala_cuajone': (-70.85, -17.30, -70.50, -16.95), 'indonesia_sorowako': (121.20, -2.65, 121.55, -2.35),
    'cirebon': (108.40, -6.90, 108.75, -6.60), 'malang': (112.50, -8.05, 112.75, -7.85),
    'tasikmalaya': (108.10, -7.45, 108.35, -7.25), 'manokwari': (133.95, -0.95, 134.20, -0.75),
    'aceh_acehtimur': (97.50, 4.20, 98.10, 4.80), 'sulbar_pasangkayu': (119.20, -1.40, 119.80, -0.80),
    'riau_tepigambut': (101.90, -0.20, 102.75, 0.70), 'ntb_sumbawa_pascabakar': (117.20, -8.85, 117.90, -8.15),
    'jatim_ladang_v2': (111.60, -7.95, 112.30, -7.55), 'bali_tabanan': (114.95, -8.55, 115.35, -8.15),
    # --- gel 5 ---
    'usa_raymine':       (-111.05, 32.95, -110.80, 33.20), 'mexico_cananea':    (-110.40, 30.85, -110.15, 31.10),
    'chile_collahuasi':  (-68.80, -21.10, -68.50, -20.80), 'peru_lasbambas':    (-72.58, -14.10, -72.28, -13.85),
    'australia_cadia':   (148.90, -33.60, 149.20, -33.30),
    'palu':    (119.82, -1.05, 120.02, -0.78), 'kendari': (122.45, -4.05, 122.65, -3.85),
    'wamena':  (138.85, -4.20, 139.05, -4.00),
    'sumut_asahan':   (99.70,  2.80, 100.00, 3.10), 'kalbar_sanggau': (110.50, 0.00, 110.80, 0.30),
    'riau_siak':      (101.80, 0.75, 102.10, 1.05),
    'riau_bengkalis': (102.00,  1.20, 102.30,  1.50), 'sumsel_ogan':    (104.00, -3.40, 104.30, -3.10),
    'jabar_subang':       (107.75, -6.70, 108.05, -6.40), 'lampung_pesawaran':  (105.05, -5.45, 105.35, -5.15),
    # --- gel 6 (final, pasca-fix) ---
    'botswana_jwaneng':       (24.50, -24.75, 24.90, -24.35),
    'safrica_witwatersrand':  (27.65, -26.45, 28.25, -25.95),
    'peru_yanacocha':         (-78.70, -7.05, -78.30, -6.65),
    'mongolia_oyutolgoi':     (106.75, 42.95, 107.15, 43.25),
    'zambia_copperbelt':      (27.65, -12.95, 28.20, -12.45),
    'drc_kolwezi':            (25.30, -10.90, 25.80, -10.40),
    'canada_sudbury':         (-81.20, 46.30, -80.70, 46.70),
    'usa_butte':              (-112.70, 45.90, -112.30, 46.30),
    'australia_olympicdam':   (136.70, -30.55, 137.10, -30.15),
    'jambikota':      (103.45, -1.75, 103.75, -1.45),
    'ternate':        (127.25, 0.65, 127.50, 0.95),
    'palangkaraya':   (113.80, -2.35, 114.10, -2.05),
    'gorontalokota':  (122.95, 0.45, 123.20, 0.70),
    'kalteng_kotawaringin': (111.35, -3.00, 111.80, -2.55),
    'jambi_merangin':       (102.15, -2.35, 102.60, -1.90),
    'aceh_nagan':           (96.25, 3.95, 96.70, 4.40),
    'kalteng_pulangpisau': (113.80, -3.10, 114.20, -2.70),
    'sumsel_oki':          (104.95, -3.45, 105.40, -3.00),
    'jambi_tanjabar':      (103.30, -1.25, 103.75, -0.80),
    'jateng_cilacap':          (108.85, -7.90, 109.30, -7.45),
    'sulsel_sidrap':           (119.75, -4.00, 120.20, -3.55),
    'lampung_tulangbawang':    (105.50, -4.55, 105.95, -4.10),
}

# Region BARU -- KHUSUS train (16 region, target ~20jt px/kelas).
TRAIN_GEL7 = {
    'tambang': [
        ('uzbekistan_muruntau',   (64.35, 41.35, 64.95, 41.70)),    # emas, tambang terbuka terbesar di dunia, sejak 1967
        ('russia_norilsk',        (87.85, 69.15, 88.55, 69.55)),    # nikel-tembaga-paladium, sejak 1930-an
        ('india_jharia',          (86.05, 23.50, 86.65, 23.90)),    # batubara, coalfield raksasa sejak abad-19
        ('indonesia_bangkatin',   (105.80, -2.35, 106.40, -1.80)),  # timah aluvial, sebaran luas
        ('sweden_kiirunavaara',   (20.05, 67.70, 20.45, 68.00)),    # bijih besi, sejak 1900
        ('china_dexing',          (117.35, 28.75, 117.90, 29.15)),  # tembaga, sejak 1955
        ('kazakhstan_zhezkazgan', (67.45, 47.60, 68.05, 48.00)),    # tembaga, sejak 1930-an
        ('mauritania_zouerate',   (-13.00, 22.50, -12.30, 22.90)),  # bijih besi raksasa
    ],
    'permukiman': [
        ('indonesia_batam',   (103.95, 0.95, 104.35, 1.35)),
        ('indonesia_tarakan', (117.55, 3.20, 117.70, 3.35)),
    ],
    'sawit': [
        ('kaltara_palm',  (117.00, 3.00, 117.45, 3.45)),
        ('bengkulu_palm', (102.15, -3.65, 102.60, -3.20)),
    ],
    'lahan_terbuka': [
        ('kalbar_sintang', (111.35, -0.05, 111.80, 0.40)),
        ('sumut_tapanuli', (98.95, 1.45, 99.40, 1.90)),
    ],
    'pertanian_lain': [
        ('sumbar_agam', (100.15, -0.45, 100.60, 0.00)),
        ('ntt_flores',  (122.15, -8.75, 122.60, -8.30)),
    ],
}

# Region BARU -- KHUSUS val/test (12 region, >=2/kelas spy bisa split PER-REGION, target
# ~10jt px/kelas GABUNGAN val+test).
VALTEST_GEL7 = {
    'tambang': [
        ('ghana_obuasi',        (-1.90, 6.05, -1.50, 6.40)),       # emas
        ('mali_loulogounkoto',  (-11.25, 12.95, -10.80, 13.35)),   # emas
        ('peru_cerrodepasco',   (-76.50, -10.85, -76.05, -10.50)), # polimetal, sejak kolonial
        ('bolivia_cerrorico',   (-65.90, -19.85, -65.50, -19.50)), # perak, sejak kolonial
    ],
    'permukiman': [
        ('indonesia_tegalpekalongan', (108.95, -7.05, 109.35, -6.75)),
        ('indonesia_cilegon',         (106.00, -6.05, 106.20, -5.85)),
    ],
    'sawit': [
        ('sultra_palm',         (121.80, -4.40, 122.25, -3.95)),
        ('sulbar_mamujutengah', (118.70, -2.50, 119.10, -2.10)),
    ],
    'lahan_terbuka': [
        ('aceh_acehbarat',   (95.75, 4.25, 96.25, 4.75)),
        ('sumsel_muaraenim', (104.50, -2.50, 104.90, -2.10)),
    ],
    'pertanian_lain': [
        ('sulut_minahasa', (125.05, 0.85, 125.45, 1.25)),
        ('ntb_lombok',     (116.20, -8.70, 116.60, -8.30)),
    ],
}


def _bbox_overlap(a, b):
    # True bila 2 bbox (lon0,lat0,lon1,lat1) berpotongan (area > 0).
    ax0, ay0, ax1, ay1 = a
    bx0, by0, bx1, by1 = b
    return ax0 < bx1 and bx0 < ax1 and ay0 < by1 and by0 < ay1


def _flatten(d):
    return [(slug, name, bbox) for slug, regions in d.items() for name, bbox in regions]


train_list = _flatten(TRAIN_GEL7)
valtest_list = _flatten(VALTEST_GEL7)
all_new = train_list + valtest_list

print('=== Cross-check 1: OVERLAP KOORDINAT vs SEMUA bbox terpakai (gelombang 1-6) ===')
geo_overlaps = []
for slug, name, bbox in all_new:
    for used_name, used_bbox in ALREADY_USED_BBOXES.items():
        if _bbox_overlap(bbox, used_bbox):
            geo_overlaps.append((slug, name, used_name))
assert not geo_overlaps, f'Region BARU overlap dgn region historis: {geo_overlaps}'
print(f'OK -- {len(all_new)} region baru TIDAK overlap koordinat dgn {len(ALREADY_USED_BBOXES)} '
      f'bbox historis (gelombang 1-6).')

print('\n=== Cross-check 2: OVERLAP internal antar region baru (train vs valtest, dst) ===')
internal_overlaps = []
for i in range(len(all_new)):
    for j in range(i + 1, len(all_new)):
        s1, n1, b1 = all_new[i]
        s2, n2, b2 = all_new[j]
        if _bbox_overlap(b1, b2):
            internal_overlaps.append((s1, n1, s2, n2))
assert not internal_overlaps, f'Region baru overlap satu sama lain: {internal_overlaps}'
print(f'OK -- {len(all_new)} region baru ({len(train_list)} train + {len(valtest_list)} valtest) '
      f'tidak overlap satu sama lain.')

import math
print(f'\nTotal region: {len(train_list)} train + {len(valtest_list)} valtest = {len(all_new)}')
for label, d in (('TRAIN', TRAIN_GEL7), ('VALTEST', VALTEST_GEL7)):
    print(f'\n--- {label}_GEL7 ---')
    for slug, regions in d.items():
        print(f'{slug}:')
        for name, bbox in regions:
            x0, y0, x1, y1 = bbox
            w_km = (x1 - x0) * 111 * math.cos(math.radians((y0 + y1) / 2))
            h_km = (y1 - y0) * 111
            print(f'  {name:<26}: {bbox}  (~{w_km:.0f} x {h_km:.0f} km)')


In [ ]:
# === Bagian T2 -- GATE: relevansi region + estimasi volume PIKSEL (target 20jt train / 10jt valtest) ===
import numpy as np
from forestwatch.gee.composite import s2_composite
from forestwatch.gee.label_fusion import build_label
from forestwatch.constants import CLASS_NAMES, CLASS_SLUGS, N_CLASSES, DEG_TO_M

EST_SCALE = 100
MIN_OWN_FRAC = 0.01
SLUG_TO_CLASS = {v: k for k, v in enumerate(CLASS_SLUGS)}
TARGET_TRAIN_PX_PER_CLASS = 20_000_000
TARGET_VALTEST_PX_PER_CLASS = 10_000_000


def _estimate_regions(regions_dict, label_tag):
    gate = {}
    reports = []
    agg_est_px = {c: 0.0 for c in range(N_CLASSES)}
    print(f'\nGATE pre-export {label_tag} (scale {EST_SCALE} m)...\n')
    for slug, regions in regions_dict.items():
        own_cls = SLUG_TO_CLASS[slug]
        for name, bbox in regions:
            x0, y0, x1, y1 = bbox
            w_m = (x1 - x0) * DEG_TO_M * np.cos(np.radians((y0 + y1) / 2))
            h_m = (y1 - y0) * DEG_TO_M
            area_px = (w_m / 10) * (h_m / 10)
            box = ee.Geometry.Rectangle(list(bbox))
            img = s2_composite(T2, box)
            label = build_label(box, T2, composite=img)
            fh = label.reduceRegion(ee.Reducer.frequencyHistogram(), box, EST_SCALE,
                                     maxPixels=int(1e13)).getInfo().get('label', {})
            cnt = {c: float(fh.get(str(c), fh.get(f'{c}.0', 0)) or 0) for c in range(N_CLASSES)}
            tot = sum(cnt.values()) or 1.0
            own_frac = cnt[own_cls] / tot
            own_px_est = area_px * own_frac
            reports.append({'slug': slug, 'name': name, 'own_class': CLASS_NAMES[own_cls],
                             'own_class_frac': own_frac, 'own_class_px_est': own_px_est})
            key = f'{slug}/{name}: >={MIN_OWN_FRAC * 100:.0f}% piksel {CLASS_NAMES[own_cls]}'
            gate[key] = own_frac >= MIN_OWN_FRAC
            print(f'  {slug:<16}/{name:<26}: {CLASS_NAMES[own_cls]:<14} frac={own_frac * 100:5.1f}% '
                  f'(~{own_px_est / 1e6:6.2f} jt px)')
            agg_est_px[own_cls] += own_px_est
    return gate, reports, agg_est_px


gate_train, reports_train, agg_train_px = _estimate_regions(TRAIN_GEL7, 'TRAIN gelombang-7')
gate_valtest, reports_valtest, agg_valtest_px = _estimate_regions(VALTEST_GEL7, 'VALTEST gelombang-7')

print('\n=== GATE T2: kelayakan region (>=1% kelas targetnya) ===')
all_gate = {**gate_train, **gate_valtest}
for k, v in all_gate.items():
    print(f"  [{'OK ' if v else 'X  '}] {k}")

print(f'\n=== Estimasi volume TRAIN-ONLY gel.7 vs target ({TARGET_TRAIN_PX_PER_CLASS / 1e6:.0f} jt px/kelas) ===')
for c in (2, 3, 4, 5, 6):
    est = agg_train_px[c]
    pct = 100 * est / TARGET_TRAIN_PX_PER_CLASS
    print(f'  {CLASS_NAMES[c]:<16}: ~{est / 1e6:8.2f} jt px  ({pct:5.1f}% target)')

print(f'\n=== Estimasi volume VAL+TEST gel.7 vs target ({TARGET_VALTEST_PX_PER_CLASS / 1e6:.0f} jt px/kelas) ===')
for c in (2, 3, 4, 5, 6):
    est = agg_valtest_px[c]
    pct = 100 * est / TARGET_VALTEST_PX_PER_CLASS
    print(f'  {CLASS_NAMES[c]:<16}: ~{est / 1e6:8.2f} jt px  ({pct:5.1f}% target)')

print('\nCatatan: ini estimasi LUAS (bukan jumlah piksel valid pasca-cut -- ada yg dibuang krn')
print('NaN/awan). Angka AKTUAL baru diketahui di T7/T7b. Kalau jauh di bawah target, tambah')
print('region lagi di T1 (pola aditif yg sama) sebelum lanjut export.')

save_json({'scale_m': EST_SCALE, 'min_own_frac': MIN_OWN_FRAC,
           'reports_train': reports_train, 'reports_valtest': reports_valtest,
           'gate_train': gate_train, 'gate_valtest': gate_valtest,
           'target_train_px_per_class': TARGET_TRAIN_PX_PER_CLASS,
           'target_valtest_px_per_class': TARGET_VALTEST_PX_PER_CLASS,
           'agg_train_px': {CLASS_NAMES[c]: agg_train_px[c] for c in range(N_CLASSES)},
           'agg_valtest_px': {CLASS_NAMES[c]: agg_valtest_px[c] for c in range(N_CLASSES)}},
          DIST_DIR / 'pixel_estimate_gel7.json')

for c in (2, 3, 4, 5, 6):
    assert agg_train_px[c] > 0, f'{CLASS_NAMES[c]} estimasi TRAIN 0 px -- region gagal, revisi T1.'
    assert agg_valtest_px[c] > 0, f'{CLASS_NAMES[c]} estimasi VALTEST 0 px -- region gagal, revisi T1.'
n_weak = sum(1 for v in all_gate.values() if not v)
if n_weak:
    print(f'\n*** PERINGATAN: {n_weak} region <1% kelas targetnya -- review/ganti bbox di T1. ***')
else:
    print('\nGATE LULUS -- semua region gelombang-7 relevan dgn kelas targetnya. Lanjut ke T3.')


In [ ]:
# === Bagian T3 -- Export GEE (TRAIN: folder train7_<slug>, VALTEST: folder valtest7_<slug>) ===
import math

assert all(_v for _v in __import__('json').load(open(DIST_DIR / 'pixel_estimate_gel7.json'))['gate_train'].values()), (
    'GATE T2 (train) belum lulus semua -- revisi region di T1 dulu.')
assert all(_v for _v in __import__('json').load(open(DIST_DIR / 'pixel_estimate_gel7.json'))['gate_valtest'].values()), (
    'GATE T2 (valtest) belum lulus semua -- revisi region di T1 dulu.')
from forestwatch.gee.tiles import make_tiles
from forestwatch.gee.export import export_tiles_grid

TARGET_KM_PER_TILE = 25


def _export_regions(regions_dict, prefix_tag, folder_tag):
    tasks = []
    for slug, regions in regions_dict.items():
        for name, bbox in regions:
            x0, y0, x1, y1 = bbox
            w_km = (x1 - x0) * 111 * math.cos(math.radians((y0 + y1) / 2))
            h_km = (y1 - y0) * 111
            nx = max(2, round(w_km / TARGET_KM_PER_TILE))
            ny = max(2, round(h_km / TARGET_KM_PER_TILE))

            box = ee.Geometry.Rectangle(list(bbox))
            img = s2_composite(T2, box)
            label = build_label(box, T2, composite=img)
            stack = img.addBands(label.toFloat())
            tiles = make_tiles(box, nx=nx, ny=ny)
            t = export_tiles_grid(
                stack, tiles,
                name_prefix=f'{prefix_tag}_{slug}_{name}_tile',
                folder=f'{folder_tag}_{slug}',
                scale=cfg['sentinel2']['scale'],
                max_pixels=int(cfg['export']['max_pixels']),
            )
            tasks.extend(t)
            print(f'  {slug:<16}/{name:<26}: {w_km:.0f}x{h_km:.0f} km -> grid {nx}x{ny} = {nx*ny} tile')
    return tasks


print('--- Submit TRAIN gelombang-7 ---')
train7_tasks = _export_regions(TRAIN_GEL7, 'train7', 'train7')
print('\n--- Submit VALTEST gelombang-7 ---')
valtest7_tasks = _export_regions(VALTEST_GEL7, 'valtest7', 'valtest7')

print(f'\n{len(train7_tasks)} task TRAIN + {len(valtest7_tasks)} task VALTEST disubmit.')
print('Pantau: https://code.earthengine.google.com/tasks')
print('Tunggu SEMUA task COMPLETED sebelum lanjut ke T4 (pindah tile nyasar + cut patches).')


In [ ]:
# === Bagian T4 -- Pindahkan tile yang nyasar (TRAIN dan VALTEST) ===
import shutil


def _move_stray(regions_dict, folder_tag, tiles_root):
    for slug in regions_dict:
        src = DRIVE_ROOT / 'Augmented_Patches' / f'{folder_tag}_{slug}'
        dst = tiles_root / slug
        dst.mkdir(parents=True, exist_ok=True)
        if src.exists():
            moved = 0
            for f in src.glob('*.tif'):
                shutil.move(str(f), str(dst / f.name))
                moved += 1
            if moved:
                print(f'{slug}: {moved} file dipindah -> {dst}')


_move_stray(TRAIN_GEL7, 'train7', TILES_GEL7_TRAIN)
_move_stray(VALTEST_GEL7, 'valtest7', TILES_GEL7_VALTEST)
print('Selesai cek tile nyasar.')


In [ ]:
# === Bagian T5 -- cut_patches_resilient (disalin verbatim dari gelombang-5/6) ===
import numpy as np
import rasterio
import shutil
import time
from rasterio.windows import Window
from pathlib import Path
from tqdm.auto import tqdm


def _remount_drive():
    from google.colab import drive
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    time.sleep(3)
    drive.mount('/content/drive', force_remount=True)
    time.sleep(2)
    print("  -> Drive di-remount.")


def _read_done_count(done_marker):
    try:
        if done_marker.exists():
            return int(done_marker.read_text().strip())
    except (OSError, ValueError):
        return None
    return None


def cut_patches_resilient(tile_dir, patch_base_dir, *,
                          patch_size=256, stride=256, max_nan_ratio=0.3,
                          n_channels_image=6, local_tmp='/content/_tmp_patches'):
    tile_files = sorted(Path(tile_dir).glob('*.tif'))
    if not tile_files:
        raise FileNotFoundError(f"Tidak ada .tif di {tile_dir}")

    patch_base = Path(patch_base_dir)
    local_root = Path(local_tmp)
    n_tiles = len(tile_files)
    total = 0

    for ti, tif in enumerate(tile_files):
        drive_out = patch_base / f'tile_{ti:03d}'
        done_marker = drive_out / '_DONE'
        header = f"Tile {ti+1}/{n_tiles}  (tile_{ti:03d})"

        try:
            done_n = _read_done_count(done_marker)
            actual_n = len(list(drive_out.glob('p*.npz'))) if drive_out.exists() else 0
        except OSError:
            _remount_drive()
            done_n = _read_done_count(done_marker)
            actual_n = len(list(drive_out.glob('p*.npz'))) if drive_out.exists() else 0

        if done_n is not None and actual_n == done_n:
            total += actual_n
            tag = "laut/kosong" if done_n == 0 else f"{actual_n} patch"
            print(f"{header}: SKIP - komplit ({tag})")
            continue

        try:
            if drive_out.exists():
                shutil.rmtree(drive_out, ignore_errors=True)
        except OSError:
            _remount_drive()
            shutil.rmtree(drive_out, ignore_errors=True)

        ltile = local_root / f'tile_{ti:03d}'
        if ltile.exists():
            shutil.rmtree(ltile)
        ltile.mkdir(parents=True, exist_ok=True)

        idx = 0
        with rasterio.open(tif) as src:
            W, H, nb = src.width, src.height, src.count
            row_list = list(range(0, H - patch_size + 1, stride))
            col_list = list(range(0, W - patch_size + 1, stride))
            pbar = tqdm(total=len(row_list) * len(col_list), desc=header, unit="win", leave=True)
            for r in row_list:
                for c in col_list:
                    arr = src.read(window=Window(c, r, patch_size, patch_size))
                    pbar.update(1)
                    if arr.shape != (nb, patch_size, patch_size):
                        continue
                    img = arr[:n_channels_image].astype('float32')
                    if np.isnan(img).mean() > max_nan_ratio:
                        continue
                    img = np.nan_to_num(img)
                    kw = dict(img=img, tile=tif.name, row=r, col=c)
                    if nb > n_channels_image:
                        kw['lab'] = np.nan_to_num(arr[n_channels_image], nan=0.0).astype('uint8')
                    np.savez_compressed(ltile / f'p{idx:05d}.npz', **kw)
                    idx += 1
                    pbar.set_postfix(patch=idx)
            pbar.close()

        local_files = sorted(ltile.glob('p*.npz'))
        assert len(local_files) == idx, "jumlah file lokal tidak konsisten"

        synced = False
        for attempt in range(1, 8):
            try:
                drive_out.mkdir(parents=True, exist_ok=True)
                for f in local_files:
                    dst = drive_out / f.name
                    if (not dst.exists()) or (dst.stat().st_size != f.stat().st_size):
                        shutil.copy2(f, dst)
                drive_n = len(list(drive_out.glob('p*.npz')))
                if drive_n == idx:
                    done_marker.write_text(str(idx))
                    synced = True
                    break
                print(f"  verifikasi belum cocok: Drive={drive_n} vs lokal={idx} (attempt {attempt})")
            except OSError as e:
                print(f"  sync gagal (attempt {attempt}): {e}")
                _remount_drive()
            time.sleep(2)
        shutil.rmtree(ltile, ignore_errors=True)

        if not synced:
            print(f"{header}: GAGAL verifikasi sync - STOP. Hapus folder tile ini di Drive lalu jalankan ulang.")
            return total

        total += idx
        tag = "LAUT/kosong (0 patch)" if idx == 0 else f"{idx} patch"
        print(f"{header}: SELESAI - {tag} -> Drive/tile_{ti:03d}/\n")

    print(f"=== SEMUA TILE SELESAI. Total {total} patch ===")
    return total


print("cut_patches_resilient siap.")


In [ ]:
# === Bagian T6 -- Jalankan cut patches (TRAIN dan VALTEST), jalankan SETELAH semua task T3 COMPLETED ===
from forestwatch.data.patches import list_patches


def _cut_all(regions_dict, tiles_root, patches_root, manifest_name):
    manifest = {'source': manifest_name, 'classes': {}}
    for slug in regions_dict:
        tile_dir  = tiles_root / slug
        patch_dir = patches_root / slug
        n_tif = len(sorted(tile_dir.glob('*.tif'))) if tile_dir.exists() else 0
        if n_tif == 0:
            print(f'  [skip] {slug}: belum ada .tif di {tile_dir} (task T3 selesai?).')
            continue
        cut_patches_resilient(tile_dir, patch_dir,
                              patch_size=cfg['patches']['size'], stride=cfg['patches']['size'])
        n_patch = len(list_patches(patch_dir))
        manifest['classes'][slug] = {'n_tif': int(n_tif), 'n_patch': int(n_patch)}
        print(f'  [ok] {slug}: {n_tif} tif -> {n_patch} patch')
    return manifest


print('--- Cut TRAIN gelombang-7 ---')
manifest_train7 = _cut_all(TRAIN_GEL7, TILES_GEL7_TRAIN, PATCHES_GEL7_TRAIN, 'train_gel7')
save_json(manifest_train7, DIST_DIR / 'train_gel7_export_manifest.json')

print('\n--- Cut VALTEST gelombang-7 ---')
manifest_valtest7 = _cut_all(VALTEST_GEL7, TILES_GEL7_VALTEST, PATCHES_GEL7_VALTEST, 'valtest_gel7')
save_json(manifest_valtest7, DIST_DIR / 'valtest_gel7_export_manifest.json')

print('\nFASE cut selesai. Manifest -> Distribution_Reports/{train,valtest}_gel7_export_manifest.json')


In [ ]:
# === Bagian T7 -- Verifikasi AKTUAL TRAIN gel.7 pasca-cut + cek vs target volume ===
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from forestwatch.constants import CLASS_NAMES, N_CLASSES

train_new7 = list_patches(PATCHES_GEL7_TRAIN)
print(f'Total patch TRAIN gelombang-7 (semua kelas gabung): {len(train_new7)}')


def _read_lab(f):
    return np.load(f)['lab']


dist_train7 = {c: 0 for c in range(N_CLASSES)}
with ThreadPoolExecutor(max_workers=32) as exe:
    for lab in tqdm(exe.map(_read_lab, train_new7), total=len(train_new7),
                    desc='Distribusi TRAIN gel.7'):
        u, cnt = np.unique(lab, return_counts=True)
        for cls, n in zip(u.tolist(), cnt.tolist()):
            if 0 <= int(cls) < N_CLASSES:
                dist_train7[int(cls)] += int(n)

print('\nDistribusi piksel TRAIN GELOMBANG-7 (AKTUAL pasca-cut) vs target 20jt/kelas:')
for c in (2, 3, 4, 5, 6):
    pct = 100 * dist_train7[c] / TARGET_TRAIN_PX_PER_CLASS
    print(f'  {CLASS_NAMES[c]:<16}: {dist_train7[c]:>12,} px  ({pct:5.1f}% target)')

save_json({'counts': {str(c): dist_train7[c] for c in range(N_CLASSES)}, 'n_patch': len(train_new7),
           'target_px_per_class': TARGET_TRAIN_PX_PER_CLASS},
          DIST_DIR / 'distribution_gel7_train.json')
print('\nVerifikasi kelima kelas (2-6) HARUS > 0 sebelum lanjut ke T7b:')
for c in (2, 3, 4, 5, 6):
    assert dist_train7[c] > 0, (
        f'{CLASS_NAMES[c]} = 0 piksel di TRAIN gel.7 -- region kelas ini gagal, revisi T1/T2.')
print('OK -- kelima kelas (2-6) terbukti punya piksel NYATA di TRAIN gelombang-7.')


In [ ]:
# === Bagian T7b -- Verifikasi AKTUAL VALTEST gel.7 + SPLIT PER-REGION jadi val_new7/test_new7 ===
# Split PER-REGION (bukan per-patch) -- setengah region pertama tiap kelas -> val, setengah
# terakhir -> test. Region identity dibaca dari field 'tile' yg tersimpan di tiap .npz (sama
# spt pola export_eval_only_transfer.ipynb), bukan dari struktur folder.
valtest7_patches = list_patches(PATCHES_GEL7_VALTEST)
print(f'Total patch VALTEST gelombang-7 (semua kelas gabung): {len(valtest7_patches)}')


def _tile_of(patch_file):
    return str(np.load(patch_file)['tile'])


def _region_of(tile_name, slug):
    candidates = sorted(VALTEST_GEL7[slug], key=lambda r: -len(r[0]))
    for name, _bbox in candidates:
        if tile_name.startswith(f'valtest7_{slug}_{name}_tile'):
            return name
    raise ValueError(f'Tile {tile_name!r} tidak cocok region manapun di slug {slug!r}')


with ThreadPoolExecutor(max_workers=32) as exe:
    tile_names = list(tqdm(exe.map(_tile_of, valtest7_patches), total=len(valtest7_patches),
                            desc='Baca field tile'))

by_slug_region = {slug: {name: [] for name, _ in regions} for slug, regions in VALTEST_GEL7.items()}
for f, tile_name in zip(valtest7_patches, tile_names):
    slug = f.relative_to(PATCHES_GEL7_VALTEST).parts[0]
    region = _region_of(tile_name, slug)
    by_slug_region[slug][region].append(f)

val_new7, test_new7 = [], []
region_assignment = {}
for slug, regions in VALTEST_GEL7.items():
    names = [n for n, _ in regions]
    half = (len(names) + 1) // 2
    val_names, test_names = names[:half], names[half:]
    region_assignment[slug] = {'val': val_names, 'test': test_names}
    for n in val_names:
        val_new7.extend(by_slug_region[slug][n])
    for n in test_names:
        test_new7.extend(by_slug_region[slug][n])

print('\nPembagian region per kelas (val vs test):')
for slug, d in region_assignment.items():
    print(f'  {slug:<16}: val={d["val"]}  test={d["test"]}')
print(f'\nval_new7={len(val_new7)} patch | test_new7={len(test_new7)} patch')

dist_val7 = {c: 0 for c in range(N_CLASSES)}
dist_test7 = {c: 0 for c in range(N_CLASSES)}
for label, files, acc in (('val_new7', val_new7, dist_val7), ('test_new7', test_new7, dist_test7)):
    with ThreadPoolExecutor(max_workers=32) as exe:
        for lab in tqdm(exe.map(_read_lab, files), total=len(files), desc=f'Distribusi {label}'):
            u, cnt = np.unique(lab, return_counts=True)
            for cls, n in zip(u.tolist(), cnt.tolist()):
                if 0 <= int(cls) < N_CLASSES:
                    acc[int(cls)] += int(n)

print('\nDistribusi piksel VAL_NEW7 + TEST_NEW7 (AKTUAL pasca-cut) vs target 10jt/kelas GABUNGAN:')
for c in (2, 3, 4, 5, 6):
    gabung = dist_val7[c] + dist_test7[c]
    pct = 100 * gabung / TARGET_VALTEST_PX_PER_CLASS
    print(f'  {CLASS_NAMES[c]:<16}: val={dist_val7[c]:>11,} px | test={dist_test7[c]:>11,} px | '
          f'gabung={gabung:>11,} px ({pct:5.1f}% target)')

save_json({'val_counts': {str(c): dist_val7[c] for c in range(N_CLASSES)},
           'test_counts': {str(c): dist_test7[c] for c in range(N_CLASSES)},
           'n_val': len(val_new7), 'n_test': len(test_new7),
           'region_assignment': region_assignment,
           'target_px_per_class_combined': TARGET_VALTEST_PX_PER_CLASS},
          DIST_DIR / 'distribution_gel7_valtest.json')

save_json({'base_dir': str(PATCHES_GEL7_VALTEST),
           'val_new7': [str(f.relative_to(PATCHES_GEL7_VALTEST)) for f in val_new7],
           'test_new7': [str(f.relative_to(PATCHES_GEL7_VALTEST)) for f in test_new7],
           'region_assignment': region_assignment},
          DIST_DIR / 'split_manifest_gel7_valtest.json')

print('\nVerifikasi kelima kelas (2-6) HARUS > 0 di VAL dan TEST sebelum lanjut ke T8:')
for c in (2, 3, 4, 5, 6):
    assert dist_val7[c] > 0, f'{CLASS_NAMES[c]} = 0 piksel di VAL gel.7 -- revisi T1/T2.'
    assert dist_test7[c] > 0, f'{CLASS_NAMES[c]} = 0 piksel di TEST gel.7 -- revisi T1/T2.'
print('OK -- kelima kelas (2-6) terbukti punya piksel NYATA di val_new7 maupun test_new7.')


In [ ]:
# === Bagian T8 -- Gabung FINAL: Papua lama + gel.4 + gel.5 + gel.6 (train) + gel.7 (train+valtest) ===
# Distribusi FIX totalnya + paket ulang .tar siap Kaggle (v4 -- mengganti Bahan_Training_Fix_Combined_v3).
from forestwatch.data.dataset import extract_dataset_archives, create_dataset_archives

# --- Muat gelombang-4 (train_new/val_new/test_new) dari manifest -- TIDAK diubah ---
_m4 = load_json(DIST_DIR / 'split_manifest_gel4_eval.json')
_base4 = Path(_m4['base_dir'])
gel4_train = [_base4 / r for r in _m4['train_new']]
gel4_val   = [_base4 / r for r in _m4['val_new']]
gel4_test  = [_base4 / r for r in _m4['test_new']]

# --- Muat gelombang-5 & gelombang-6 (train-only) -- TIDAK diubah ---
train_new_gel5 = list_patches(DRIVE_ROOT / 'ForestWatch_Patches_TrainOnlyGel5')
train_new_gel6 = list_patches(DRIVE_ROOT / 'ForestWatch_Patches_TrainOnlyGel6')

# --- Muat Papua lama (FROZEN) ---
BAHAN_DIR = DRIVE_ROOT / 'Bahan_Training_Fix'
LOCAL_OLD = Path('/content/dataset_local_old')
old_dirs = extract_dataset_archives(BAHAN_DIR, LOCAL_OLD, splits=('train', 'val', 'test'), max_workers=8)
old_train = list_patches(old_dirs['train'])
old_val   = list_patches(old_dirs['val'])
old_test  = list_patches(old_dirs['test'])

# --- Gabungan FINAL ---
final_train_files = old_train + gel4_train + train_new_gel5 + train_new_gel6 + train_new7
val_p  = old_val + gel4_val + val_new7
test_p = old_test + gel4_test + test_new7
print('Gabungan FINAL --')
print(f'  train: {len(old_train)} (Papua) + {len(gel4_train)} (gel.4) + {len(train_new_gel5)} (gel.5) + '
      f'{len(train_new_gel6)} (gel.6) + {len(train_new7)} (gel.7) = {len(final_train_files)}')
print(f'  val  : {len(old_val)} (Papua) + {len(gel4_val)} (gel.4) + {len(val_new7)} (gel.7) = {len(val_p)}')
print(f'  test : {len(old_test)} (Papua) + {len(gel4_test)} (gel.4) + {len(test_new7)} (gel.7) = {len(test_p)}')


def _dist_of(files, desc):
    counts = {c: 0 for c in range(N_CLASSES)}
    with ThreadPoolExecutor(max_workers=32) as exe:
        for lab in tqdm(exe.map(_read_lab, files), total=len(files), desc=desc):
            u, cnt = np.unique(lab, return_counts=True)
            for cls, n in zip(u.tolist(), cnt.tolist()):
                if 0 <= int(cls) < N_CLASSES:
                    counts[int(cls)] += int(n)
    return counts


print('\n=== Distribusi piksel FINAL (Papua + gel.4 + gel.5 + gel.6 + gel.7) per split ===')
dist_summary = {}
for split_name, files in (('train', final_train_files), ('val', val_p), ('test', test_p)):
    dist = _dist_of(files, desc=f'Distribusi {split_name}')
    tot = sum(dist.values()) or 1
    dist_summary[split_name] = {'n_patch': len(files), 'pixel_counts': dist}
    print(f'\n{split_name.upper()} ({len(files)} patch):')
    for c in range(N_CLASSES):
        print(f'  {CLASS_NAMES[c]:<16}: {dist[c]:>14,} px  ({100 * dist[c] / tot:5.2f}%)')

save_json(dist_summary, DIST_DIR / 'distribution_combined_papua_plus_gel4_gel5_gel6_gel7.json')

# --- Paket jadi .tar -- v4, siap upload sbg Kaggle Dataset ---
COMBINED_DIR = DRIVE_ROOT / 'Bahan_Training_Fix_Combined_v4'
splits_for_archive = {
    'train': [(f'p{i:06d}.npz', f) for i, f in enumerate(final_train_files)],
    'val':   [(f'p{i:06d}.npz', f) for i, f in enumerate(val_p)],
    'test':  [(f'p{i:06d}.npz', f) for i, f in enumerate(test_p)],
}
archives = create_dataset_archives(splits_for_archive, COMBINED_DIR, n_train_parts=7, max_workers=16)
print(f'\nArsip gabungan FINAL -> {COMBINED_DIR}')
for split_name, paths in archives.items():
    print(f'  {split_name}: {[p.name for p in paths]}')
print('\nLangkah Kaggle: upload folder Bahan_Training_Fix_Combined_v4/ sbg Kaggle Dataset BARU')
print('(superseded v3 -- pakai v4 ini sekarang), lalu attach ke improve_model.ipynb (ENV="kaggle").')
